# EDA und Datenvorbereitung

Ziel dieses Notebooks ist es, die Data-Collection-Outputs zu verstehen, erste Qualitätschecks durchzuführen und die Datasets für die Analyse vorzubereiten

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px

In [2]:
DATA_DIR = Path("../../data")
INTERIM_DIR = DATA_DIR / "interim"

In [3]:
COSTS_FILE = INTERIM_DIR / "gesundheitskosten_2011_2026.csv"

df_costs = pd.read_csv(COSTS_FILE)

print(pd.DataFrame({
        "rows": [len(df_costs)],
        "columns": [df_costs.shape[1]],
        "duplicate_rows": [df_costs.duplicated().sum()],
        "missing_cells": [df_costs.isna().sum().sum()],
}))
empty_columns = df_costs.columns[df_costs.isna().all()].tolist()
print(empty_columns)

   rows  columns  duplicate_rows  missing_cells
0  7372       42               0          44232
['Time Period', 'Observation value (DotStat)', 'Date of spatial reference', 'Update date', 'Date of publication', 'Database state']


Die Gesundheitskosten enthalten 44’232 fehlende Werte. Diese entstehen  durch sechs optionale Metadatenspalten die leer sind. Die analyse-relevanten Spalten wie Jahr, Kanton, Alter, Beobachtungswert, Multiplikator und Status enthalten keine fehlenden Werte.

Es sind ebenfalls viele Spalten vorhanden, welche für die Analyse später nicht relevant sind.

In [4]:
cost_columns = [
    "TIME_PERIOD",
    "CANTON",
    "Swiss cantons",
    "AGE",
    "Age groups",
    "OBS_VALUE",
    "MULT",
    "OBS_STATUS",
    "Code list for Observation Status",
]

df_costs_processed = df_costs[cost_columns].copy()
df_costs_processed.head(30)

,TIME_PERIOD,CANTON,Swiss cantons,AGE,Age groups,OBS_VALUE,MULT,OBS_STATUS,Code list for Observation Status
0,2011,_T,Total,_T,Total,64234.604,6,A,Normal value
1,2012,_T,Total,_T,Total,66521.482,6,A,Normal value
2,2013,_T,Total,_T,Total,69352.477,6,A,Normal value
3,2014,_T,Total,_T,Total,71055.556,6,A,Normal value
4,2015,_T,Total,_T,Total,73345.969,6,A,Normal value
5,2016,_T,Total,_T,Total,75935.877,6,A,Normal value
6,2017,_T,Total,_T,Total,77866.583,6,A,Normal value
7,2018,_T,Total,_T,Total,79171.643,6,A,Normal value
8,2019,_T,Total,_T,Total,81686.113,6,A,Normal value
9,2020,_T,Total,_T,Total,83522.010,6,A,Normal value


In [5]:
print(pd.DataFrame({
        "year_min": [df_costs_processed["TIME_PERIOD"].min()],
        "year_max": [df_costs_processed["TIME_PERIOD"].max()],
        "n_years": [df_costs_processed["TIME_PERIOD"].nunique()],
        "n_cantons": [df_costs_processed["Swiss cantons"].nunique()],
        "n_age_groups": [df_costs_processed["Age groups"].nunique()],
        "n_status": [df_costs_processed["OBS_STATUS"].nunique()],
        "n_multipliers": [df_costs_processed["MULT"].nunique()],
}))

   year_min  year_max  n_years  n_cantons  n_age_groups  n_status  \
0      2011      2024       14         27            21         3   

   n_multipliers  
0              1  


In [6]:
print(f'Status: \n {df_costs_processed["OBS_STATUS"].unique()}')
print(f'Kantone: \n {df_costs_processed["Swiss cantons"].unique()}')

Status: 
 <StringArray>
['A', 'P', 'E']
Length: 3, dtype: str
Kantone: 
 <StringArray>
[                 'Total',                 'Zurich',                   'Bern',
                'Lucerne',                    'Uri',                 'Schwyz',
               'Obwalden',              'Nidwalden',                 'Glarus',
                    'Zug',               'Fribourg',              'Solothurn',
            'Basel-Stadt',       'Basel-Landschaft',           'Schaffhausen',
 'Appenzell Ausserrhoden',  'Appenzell Innerrhoden',             'St. Gallen',
             'Graubünden',                 'Aargau',                'Thurgau',
                 'Ticino',                   'Vaud',                 'Valais',
              'Neuchâtel',                 'Geneva',                   'Jura']
Length: 27, dtype: str


In [7]:
pd.crosstab(
    df_costs_processed["TIME_PERIOD"],
    df_costs_processed["OBS_STATUS"]
)

OBS_STATUS,A,E,P
TIME_PERIOD,,,
2011,567,0,0
2012,567,0,0
2013,567,0,0
2014,567,0,0
2015,567,0,0
2016,567,0,0
2017,567,0,0
2018,567,0,0
2019,567,0,0


Für den Status haben 3 verschiedene Werte:

- A: Normaler Wert
- E: Geschätzter Wert
- P: Provisorischer Wert

Das Jahr 2023 beinhaltet nur provisorische Werte für alle Zeilen.
Das Jahr 2024 beinhaltet insgesamt nur einen geschätzten Wert.

Die Spalte Kantone beinhaltet alle 26 Kantone und das Total für die Schweiz

Um den absolut Wert für die Kosten zu erhalten müssen wird eine zusätzliche Spalte einfügen

In [8]:
df_costs_processed["costs_chf"] = df_costs["OBS_VALUE"] * (10 ** df_costs["MULT"])
df_costs_processed.head()

,TIME_PERIOD,CANTON,Swiss cantons,AGE,Age groups,OBS_VALUE,MULT,OBS_STATUS,Code list for Observation Status,costs_chf
0,2011,_T,Total,_T,Total,64234.604,6,A,Normal value,6.423460e+10
1,2012,_T,Total,_T,Total,66521.482,6,A,Normal value,6.652148e+10
2,2013,_T,Total,_T,Total,69352.477,6,A,Normal value,6.935248e+10
3,2014,_T,Total,_T,Total,71055.556,6,A,Normal value,7.105556e+10
4,2015,_T,Total,_T,Total,73345.969,6,A,Normal value,7.334597e+10


In [9]:
df_costs_total_ch = df_costs_processed[
    (df_costs_processed["Swiss cantons"] == "Total") &
    (df_costs_processed["Age groups"] == "Total")
].copy()

px.line(
    df_costs_total_ch,
    x="TIME_PERIOD",
    y="costs_chf",
    markers=True,
    title="Gesundheitskosten Schweiz, Total",
    labels={"TIME_PERIOD": "Jahr", "costs_chf": "Kosten in CHF"},
)

In [10]:
df_costs_canton_total = df_costs_processed[
    (df_costs_processed["Age groups"] == "Total") &
    (df_costs_processed["Swiss cantons"] != "Total")
].copy()

latest_canton_year = int(df_costs_canton_total["TIME_PERIOD"].max())

df_costs_canton_latest = (
    df_costs_canton_total[df_costs_canton_total["TIME_PERIOD"] == latest_canton_year]
    .sort_values("costs_chf", ascending=False)
)

px.bar(
    df_costs_canton_latest.head(10),
    x="Swiss cantons",
    y="costs_chf",
    title=f"Top 10 Kantone nach Gesundheitskosten {latest_canton_year}",
)

Im Plot für die Top 10 Kantone ist zu sehen, dass die Bevölkerungsgrösse eine Rolle spielt.

Für die spätere Analyse könnten die pro Kopf Kosten einen besseren Einblick geben

In [11]:
df_costs_age_ch = df_costs_processed[
    (df_costs_processed["Swiss cantons"] == "Total") &
    (df_costs_processed["Age groups"] != "Total") &
    (df_costs_processed["TIME_PERIOD"] == 2023)
].copy()

df_costs_age_ch["cost_share"] = (
    df_costs_age_ch["costs_chf"] / df_costs_age_ch["costs_chf"].sum()
)

px.bar(
    df_costs_age_ch,
    x="Age groups",
    y="cost_share",
    title="Kostenanteil nach Altersgruppe Schweiz 2023",
)

In [ ]:
PROCESSED_DIR = DATA_DIR / "processed"
PROCESSED_DIR.mkdir(exist_ok=True)

df_costs_processed.to_csv(PROCESSED_DIR / "gesundheitskosten.csv")